In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

In [32]:
tracks = pd.read_csv(
    "../data/fma_metadata/tracks.csv",
    index_col=0,
    header=[0, 1, 2]
)

features = pd.read_csv(
    "../data/fma_metadata/features.csv",
    index_col=0,
    header=[0, 1, 2]
)

In [33]:
#checking for missing vals and if data looks as expected
genre_col = ('track', 'genre_top', 'Unnamed: 40_level_2')

model_data = tracks[[genre_col]].copy()
model_data = model_data.dropna(subset=[genre_col])

mean_features = features.loc[
    :, features.columns.get_level_values(1) == 'mean'
]

model_data = model_data.join(mean_features)

print(model_data.shape)
print(model_data.head())
print(model_data.isna().sum().sum())

(49598, 75)
                  track chroma_cens                                          \
              genre_top        mean                                           
    Unnamed: 40_level_2          01        02        03        04        05   
2               Hip-Hop    0.474300  0.369816  0.236119  0.228068  0.222830   
3               Hip-Hop    0.232784  0.229469  0.225674  0.216713  0.220512   
5               Hip-Hop    0.258420  0.303385  0.250737  0.218562  0.245743   
10                  Pop    0.229882  0.286978  0.240096  0.226792  0.192443   
134             Hip-Hop    0.249513  0.253561  0.262994  0.212424  0.246259   

                                             ... spectral_contrast             \
                                             ...              mean              
           06        07        08        09  ...                06         07   
2    0.221415  0.229238  0.248795  0.196245  ...         17.616112  38.268646   
3    0.242744  0.369235  0.4207

In [34]:
# finding genre column
genre_cols = [
    col for col in tracks.columns
    if "genre_top" in col
]

print(genre_cols)

# counting tracks in each genre
genre_counts = tracks[genre_cols[0]].value_counts()

print(genre_counts)

[('track', 'genre_top', 'Unnamed: 40_level_2')]
(track, genre_top, Unnamed: 40_level_2)
Rock                   14182
Experimental           10608
Electronic              9372
Hip-Hop                 3552
Folk                    2803
Pop                     2332
Instrumental            2079
International           1389
Classical               1230
Jazz                     571
Old-Time / Historic      554
Spoken                   423
Country                  194
Soul-RnB                 175
Blues                    110
Easy Listening            24
Name: count, dtype: int64


In [35]:
X = model_data.drop(columns=[genre_col])
y = model_data[genre_col]

#flatten for simpler naming
X.columns = ['_'.join(col) for col in X.columns]

#80/20 train test split for now
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=7,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining genre distribution:")
print(y_train.value_counts())

Training set: (39678, 74)
Testing set: (9920, 74)

Training genre distribution:
(track, genre_top, Unnamed: 40_level_2)
Rock                   11346
Experimental            8486
Electronic              7498
Hip-Hop                 2842
Folk                    2242
Pop                     1866
Instrumental            1663
International           1111
Classical                984
Jazz                     457
Old-Time / Historic      443
Spoken                   338
Country                  155
Soul-RnB                 140
Blues                     88
Easy Listening            19
Name: count, dtype: int64


### Train

In [36]:
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=7)
)

logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](16,)","['Blues','Classical','Country',...,'Rock','Soul-RnB','Spoken']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](74,)","['chroma_cens_mean_01','chroma_cens_mean_02','chroma_cens_mean_03',..., 'tonnetz_mean_05','tonnetz_mean_06','zcr_mean_01']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,74
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


### Evaluate

In [37]:
lr_pred = logistic_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy:", lr_accuracy)
print(classification_report(y_test, lr_pred, zero_division=0))

Logistic Regression Accuracy: 0.5702620967741936
                     precision    recall  f1-score   support

              Blues       0.00      0.00      0.00        22
          Classical       0.64      0.67      0.66       246
            Country       0.00      0.00      0.00        39
     Easy Listening       0.00      0.00      0.00         5
         Electronic       0.55      0.60      0.57      1874
       Experimental       0.52      0.60      0.56      2122
               Folk       0.51      0.49      0.50       561
            Hip-Hop       0.53      0.34      0.41       710
       Instrumental       0.36      0.15      0.21       416
      International       0.53      0.18      0.27       278
               Jazz       0.33      0.01      0.02       114
Old-Time / Historic       0.89      0.88      0.89       111
                Pop       0.00      0.00      0.00       466
               Rock       0.63      0.83      0.72      2836
           Soul-RnB       0.00     

Comparing to Decision Tree results:
- Accuracy: + 7.54 percentage points better than Decision Tree
- Macro F1: + 0.07, suggesting better performance across genres overall, including smaller classes
- Weighted F1: + 0.08, indicating better performance when accounting for each genre's number of examples

### Try model w balanced class weight
Because the dataset is highly imbalanced, the model has a tendency to perform well on the more common genres bc they contribute more to its overall loss, so balancing gives more importance to the underrepresented genres like Easy Listening.

In [38]:
# try to use balanced class_weight
logistic_model_balanced = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        random_state=7,
        class_weight='balanced')
)

logistic_model_balanced.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](16,)","['Blues','Classical','Country',...,'Rock','Soul-RnB','Spoken']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](74,)","['chroma_cens_mean_01','chroma_cens_mean_02','chroma_cens_mean_03',..., 'tonnetz_mean_05','tonnetz_mean_06','zcr_mean_01']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,74
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


#### Evaluate Balanced

In [39]:
balanced_pred = logistic_model_balanced.predict(X_test)

balanced_accuracy = accuracy_score(y_test, balanced_pred)

print("Balanced Logistic Regression Accuracy:", balanced_accuracy)
print(classification_report(y_test, balanced_pred, zero_division=0))

Balanced Logistic Regression Accuracy: 0.4184475806451613
                     precision    recall  f1-score   support

              Blues       0.02      0.36      0.04        22
          Classical       0.44      0.74      0.55       246
            Country       0.05      0.64      0.08        39
     Easy Listening       0.01      0.40      0.01         5
         Electronic       0.64      0.35      0.46      1874
       Experimental       0.63      0.30      0.41      2122
               Folk       0.41      0.39      0.40       561
            Hip-Hop       0.38      0.48      0.43       710
       Instrumental       0.21      0.28      0.24       416
      International       0.25      0.36      0.29       278
               Jazz       0.07      0.25      0.11       114
Old-Time / Historic       0.68      0.95      0.79       111
                Pop       0.10      0.06      0.08       466
               Rock       0.79      0.57      0.66      2836
           Soul-RnB       

Analysis:
- class weighting created a tradeoff between overall accuracy and performance across the less common genres
- macro recall improved substantially, but macro F1 decrease slightly
- so, model may identify the less-common genres more, its precision decreases

### Feature-group experiements

#### MFCC-only experiment
MFCCs primarily capture timbral characteristics of music. So, isolating to see if timbre helps classify genres.

In [40]:
mfcc_features = features.loc[
    :, ('mfcc', 'mean', slice(None))
]

# combine mfcc features w genre labels
mfcc_data = tracks[[genre_col]].copy()
mfcc_data = mfcc_data.dropna(subset=[genre_col])

mfcc_data = mfcc_data.join(mfcc_features)

# should have no missing vals
print(mfcc_data.shape)
print(mfcc_data.isna().sum().sum())

(49598, 21)
0


In [41]:
X_mfcc = mfcc_data.drop(columns=[genre_col])
y_mfcc = mfcc_data[genre_col]

X_mfcc.columns = ['_'.join(col) for col in X_mfcc.columns]

In [42]:
X_train_mfcc, X_test_mfcc, y_train_mfcc, y_test_mfcc = train_test_split(
    X_mfcc,
    y_mfcc,
    test_size=0.2,
    random_state=7,
    stratify=y_mfcc
)

Train MFCC model

In [43]:
mfcc_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        random_state=7
    )
)

mfcc_model.fit(X_train_mfcc, y_train_mfcc)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](16,)","['Blues','Classical','Country',...,'Rock','Soul-RnB','Spoken']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['mfcc_mean_01','mfcc_mean_02','mfcc_mean_03',...,'mfcc_mean_18', 'mfcc_mean_19','mfcc_mean_20']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


Evaluate MFCC model

In [44]:
mfcc_pred = mfcc_model.predict(X_test_mfcc)

print("MFCC-only Accuracy:", accuracy_score(y_test_mfcc, mfcc_pred))
print(classification_report(y_test_mfcc, mfcc_pred, zero_division=0))

MFCC-only Accuracy: 0.4998991935483871
                     precision    recall  f1-score   support

              Blues       0.00      0.00      0.00        22
          Classical       0.48      0.39      0.43       246
            Country       0.00      0.00      0.00        39
     Easy Listening       0.00      0.00      0.00         5
         Electronic       0.48      0.57      0.52      1874
       Experimental       0.40      0.51      0.45      2122
               Folk       0.42      0.25      0.31       561
            Hip-Hop       0.51      0.20      0.29       710
       Instrumental       0.21      0.02      0.04       416
      International       0.67      0.08      0.14       278
               Jazz       0.00      0.00      0.00       114
Old-Time / Historic       0.84      0.80      0.82       111
                Pop       0.00      0.00      0.00       466
               Rock       0.57      0.81      0.67      2836
           Soul-RnB       0.00      0.00     

Analysis:
- MFCC-only accuracy is 49% compared to 57% with all the features
- this suggests MFCC's alone are pretty informative, achieve almost 50% accuracy with just 20 features compared to 57% with all 74
- Rock is still relatively easy to identify. It has a F1-score of 0.67 compared to 0.72 for the full-feature model.
- Old-Time/Historic performs particularly well. It has a F1-score of 0.82 compared to 0.89 for the full-feature model.
- some genres are still difficult: Pop, Jazz, Blues, and Soul-RnB still have zero F1-score

#### Chroma-only experiment
Chroma features cpature tonal and harmonic info, representing the distribution of energy across the 12 musical pitch classes. Testing to see if Chroma helps classify genres.

In [45]:
chroma_features = features.loc[
    :, (
        ['chroma_cens', 'chroma_cqt', 'chroma_stft'],
        'mean',
        slice(None)
    )
]

#combine chroma features w genre labels
chroma_data = tracks[[genre_col]].copy()
chroma_data = chroma_data.dropna(subset=[genre_col])

chroma_data = chroma_data.join(chroma_features)

# make sure no missing vals
print(chroma_data.shape)
print(chroma_data.isna().sum().sum())

(49598, 37)
0


In [46]:
X_chroma = chroma_data.drop(columns=[genre_col])
y_chroma = chroma_data[genre_col]

X_chroma.columns = ['_'.join(col) for col in X_chroma.columns]

In [47]:
X_train_chroma, X_test_chroma, y_train_chroma, y_test_chroma = train_test_split(
    X_chroma,
    y_chroma,
    test_size=0.2,
    random_state=7,
    stratify=y_chroma
)

Train chroma model

In [48]:
chroma_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        random_state=7
    )
)

chroma_model.fit(X_train_chroma, y_train_chroma)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](16,)","['Blues','Classical','Country',...,'Rock','Soul-RnB','Spoken']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](36,)","['chroma_cens_mean_01','chroma_cens_mean_02','chroma_cens_mean_03',..., 'chroma_stft_mean_10','chroma_stft_mean_11','chroma_stft_mean_12']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,36
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


Evaluate chroma model

In [49]:
chroma_pred = chroma_model.predict(X_test_chroma)

chroma_accuracy = accuracy_score(y_test_chroma, chroma_pred)

print("Chroma-only Accuracy:", chroma_accuracy)
print(classification_report(y_test_chroma, chroma_pred, zero_division=0))

Chroma-only Accuracy: 0.41300403225806454
                     precision    recall  f1-score   support

              Blues       0.00      0.00      0.00        22
          Classical       0.21      0.11      0.14       246
            Country       0.00      0.00      0.00        39
     Easy Listening       0.00      0.00      0.00         5
         Electronic       0.38      0.37      0.37      1874
       Experimental       0.40      0.50      0.44      2122
               Folk       0.25      0.17      0.20       561
            Hip-Hop       0.18      0.02      0.03       710
       Instrumental       0.28      0.04      0.07       416
      International       0.00      0.00      0.00       278
               Jazz       0.00      0.00      0.00       114
Old-Time / Historic       0.53      0.62      0.57       111
                Pop       0.00      0.00      0.00       466
               Rock       0.46      0.75      0.57      2836
           Soul-RnB       0.00      0.00  

Analysis:
- chroma-only performed worse than mfcc-only with 41.3% accuracy compared to mfcc's 49.99%
- chroma-only struggled with many genres: zero recall for Blues, Country, Easy Listening, International, Jazz, Pop, Soul-RnB, and Spoken; but, performed relatively better for Rock (0.57 F1), Old-Time/Historic (0.57 F1) and Experimental (0.44 F1)

### Comparing results of feature experiments
- Multiple audio features improves genre classification. The model with all the features performed the best, which suggests that combining different types of audio characteristics provides more useful information than relying on one feature family itself.
- Because MFCC-only performed better than Chroma-only performed, this shows that timbral characteristics captured by MFCCs are more useful for distinguishing genres than tonal and harmonic characteristics captured by Chroma features (at least with this model for this specific dataset)
